<div style='background:linear-gradient(135deg,#3a1a5c 0%,#7d2d8f 100%);color:white;padding:22px 30px;border-radius:10px;font-family:sans-serif'>
<h1 style='margin:0 0 6px 0;font-size:1.7em'>🔑 DESAFÍO RELÁMPAGO — Sesión 10 · SOLUCIONES</h1>
<h2 style='margin:0 0 12px 0;font-weight:300;font-size:1.15em'>Karplus-Strong — versión profesor</h2>
<p style='margin:0;opacity:0.92;font-size:0.97em;line-height:1.5'>
Versión resuelta del notebook. Cada bloque clave tiene <code style='background:#000;padding:1px 4px;border-radius:3px'>[NOTA PROFESOR]</code> con: <em>qué deben oír</em>, <em>el error típico</em>, y <em>cuándo intervenir</em>.<br><br>
<strong>Plan de tiempo (30 min):</strong>
  <span style='display:inline-block;margin:0 4px;padding:1px 8px;background:rgba(255,255,255,0.15);border-radius:4px'>0–5' setup + predicción</span>
  <span style='display:inline-block;margin:0 4px;padding:1px 8px;background:rgba(255,255,255,0.15);border-radius:4px'>5–15' generar y oír beating</span>
  <span style='display:inline-block;margin:0 4px;padding:1px 8px;background:rgba(255,255,255,0.15);border-radius:4px'>15–22' fix + verificación</span>
  <span style='display:inline-block;margin:0 4px;padding:1px 8px;background:rgba(255,255,255,0.15);border-radius:4px'>22–28' escala mayor + ρ</span>
  <span style='display:inline-block;margin:0 4px;padding:1px 8px;background:rgba(255,255,255,0.15);border-radius:4px'>28–30' cierre</span>
</p></div>

## 0 · Setup — el algoritmo Karplus-Strong en 5 líneas 🪕

Karplus-Strong (KS, 1983) es probablemente el algoritmo de síntesis más elegante que vas a ver: con un
**buffer circular** y un **filtro promedio** sintetiza una cuerda pulsada que suena sorprendentemente real.

$$
\boxed{\;y[n] \;=\; \rho \cdot \tfrac{1}{2}\big(\,y[n-N] + y[n-N-1]\,\big)\;}
$$

| símbolo | qué controla | valor típico |
|---|---|---|
| $N$ | largo del buffer (cuerda virtual) | $\approx f_s/f_0$ |
| $\rho$ | factor de pérdida ("damping" / longitud de la cuerda) | 0.99–0.999 |
| buffer inicial | "el pluck" — energía que excita la cuerda | ruido blanco $\mathcal{U}(-1, 1)$ |

> **La idea física:** el buffer es una cuerda discreta de $N$ muestras. La onda recorre el buffer y vuelve.
> Cada vuelta, el filtro promedio (un LP suave) quita energía a las altas frecuencias —
> exactamente como una cuerda real pierde primero los armónicos agudos y termina sonando más senoidal.

**El bug que vas a encontrar hoy** vive en esta fórmula. Hay un detalle sutil sobre el delay total del lazo
que mete medio sample de diferencia entre $N$ y la frecuencia real producida. Ese medio sample es la
diferencia entre afinado y desafinado.

In [ ]:
%matplotlib inline
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import Audio, display
import wave, os
np.random.seed(10)  # [WHY] reproducibilidad — todos parten del mismo pluck

FS = 44100  # tasa del curso

def save_wav(filename, audio, fs=FS):
    """Guarda audio mono o estéreo Nx2 como WAV PCM int16."""
    audio = np.asarray(audio, dtype=np.float64)
    peak = np.max(np.abs(audio)) + 1e-12
    audio = audio / peak * 0.85
    pcm = np.clip(audio * 32767, -32768, 32767).astype(np.int16)
    n_channels = 2 if audio.ndim == 2 else 1
    if n_channels == 2: pcm = pcm.reshape(-1, 2)
    with wave.open(filename, 'w') as wf:
        wf.setnchannels(n_channels); wf.setsampwidth(2); wf.setframerate(fs)
        wf.writeframes(pcm.tobytes())
    print(f'  Guardado: {filename}  ({os.path.getsize(filename)/1024:.0f} KB)')

def play(audio, fs=FS):
    a = np.asarray(audio, dtype=np.float64)
    a = a / (np.max(np.abs(a)) + 1e-12) * 0.85
    if a.ndim == 2: a = a.T
    return Audio(a, rate=fs)

def plot_spec(audio, fs=FS, fmax=4000, title=''):
    N = len(audio); seg = audio[N//4 : N//4 + 8192]
    win = np.hanning(len(seg))
    X = np.fft.rfft(seg * win)
    f = np.fft.rfftfreq(len(seg), 1/fs)
    mag_db = 20*np.log10(np.abs(X) + 1e-9)
    plt.figure(figsize=(9, 2.6))
    plt.plot(f, mag_db, lw=0.9)
    plt.xlim(0, fmax); plt.ylim(np.max(mag_db)-70, np.max(mag_db)+5)
    plt.xlabel('Hz'); plt.ylabel('dB'); plt.title(title); plt.grid(alpha=0.3)
    plt.tight_layout(); plt.show()

def plot_envelope(audio, fs=FS, title=''):
    """Plotea la envolvente RMS (ventana de 20 ms) — para ver el beating."""
    win = int(0.02 * fs)
    pad = (-len(audio)) % win
    a = np.concatenate([audio, np.zeros(pad)])
    rms = np.sqrt((a.reshape(-1, win) ** 2).mean(axis=1))
    t = np.arange(len(rms)) * win / fs
    plt.figure(figsize=(9, 2.2))
    plt.plot(t, rms, lw=1.0)
    plt.xlabel('t (s)'); plt.ylabel('RMS'); plt.title(title); plt.grid(alpha=0.3)
    plt.tight_layout(); plt.show()


## 1 · La versión buggy 🐛

Esta función implementa KS de forma "obvia": calcula $N = \mathrm{round}(f_s/f_0)$, llena el buffer con
ruido y aplica el filtro promedio. Compila, no tiene errores de runtime, suena a "cuerda".

**Pero está desafinada.** Cuando le pides 440 Hz, devuelve algo cercano pero no exacto. La pregunta es:
¿cuántos Hz fuera, y por qué?

In [ ]:
def karplus_strong_buggy(f0, dur, fs=FS, rho=0.996):
    """
    Versión 'natural' del estudiante: N = round(fs/f0).
    Funciona... pero está afinada mal por una cantidad pequeña pero detectable.
    """
    N = int(round(fs / f0))
    buf = np.random.uniform(-1, 1, N).astype(np.float64)
    n_out = int(dur * fs)
    out = np.zeros(n_out)
    ptr = 0
    for i in range(n_out):
        out[i] = buf[ptr]
        # filtro promedio + factor de pérdida ρ, alimentado de vuelta al buffer
        next_sample = rho * 0.5 * (buf[ptr] + buf[(ptr - 1) % N])
        buf[ptr] = next_sample
        ptr = (ptr + 1) % N
    return out

# Genera la nota La (440 Hz) con la versión buggy
ks_buggy = karplus_strong_buggy(440.0, dur=2.0)
save_wav('desafio10_ks_buggy_440Hz.wav', ks_buggy)
play(ks_buggy)


### 1.1 — Diagnóstico por superposición (la técnica del beating)

Cuando dos tonos de frecuencias muy cercanas $f_1$ y $f_2$ se suman, el oído percibe **un solo tono**
modulado en amplitud por una envolvente lenta de $|f_1 - f_2|$ Hz. Eso es el **beating**.

Si tu KS *creyera* estar a 440 Hz pero en realidad está a 437 Hz, al sumarlo con una sinusoide pura de
440 Hz oirás una pulsación de 3 Hz. Es un velocímetro de afinación brutalmente preciso.

**Tarea:** genera la sinusoide de referencia, súmala con tu `ks_buggy` y escucha el resultado.

In [ ]:
# ── Sinusoide de referencia a 440 Hz
t = np.arange(int(2.0 * FS)) / FS
ref_440 = 0.5 * np.sin(2 * np.pi * 440 * t)

# Mezcla: KS buggy + seno de referencia
mix_buggy = ks_buggy + ref_440
save_wav('desafio10_beating_buggy_vs_ref.wav', mix_buggy)

# Visualizamos la envolvente RMS — la pulsación se ve a simple vista
plot_envelope(mix_buggy, title='Mezcla KS buggy + seno 440 Hz — ¿ves la pulsación?')
play(mix_buggy)


### Diagnóstico esperado para `ks_buggy`

Con $f_s = 44100$ y $f_0 = 440$:
- $N = \mathrm{round}(44100/440) = 100$.
- **Lo que el algoritmo realmente toca:** $f_{\text{real}} = f_s / (N + 0.5) = 44100/100.5 \approx 438.81$ Hz.
- **Desafinación:** $440 - 438.81 = 1.19$ Hz.
- **Beating esperado** entre `ks_buggy` y la referencia de 440: ≈ **1.2 Hz** (una pulsación cada ~0.84 s).
- En 2 segundos de audio oirás **2 pulsaciones completas**, perfectamente audibles.

> [NOTA PROFESOR] Si los alumnos no oyen el beating al primer intento, sugerir audífonos y subir
> volumen. La envolvente RMS lo hace evidente visualmente. Algunos van a confundir el "decay natural"
> de KS con la pulsación — explicar la diferencia: decay es monótono, beating es oscilatorio.


## 3 · Implementa la versión corregida 🔧

Implementa `karplus_strong_correct` con la compensación del medio sample. Lo único que cambia respecto
a la buggy es el cálculo de $N$.

In [ ]:
def karplus_strong_correct(f0, dur, fs=FS, rho=0.996):
    """
    Compensación del +0.5 sample del filtro promedio.
    Queremos: fs / (N + 0.5) == f0  ⇒  N = round(fs/f0 - 0.5)
    """
    N = int(round(fs / f0 - 0.5))
    buf = np.random.uniform(-1, 1, N).astype(np.float64)
    n_out = int(dur * fs)
    out = np.zeros(n_out)
    ptr = 0
    for i in range(n_out):
        out[i] = buf[ptr]
        next_sample = rho * 0.5 * (buf[ptr] + buf[(ptr - 1) % N])
        buf[ptr] = next_sample
        ptr = (ptr + 1) % N
    return out

ks_correct = karplus_strong_correct(440.0, dur=2.0)
save_wav('desafio10_ks_correct_440Hz.wav', ks_correct)
play(ks_correct)

# [NOTA PROFESOR] Sutilezas que algunos alumnos descubren:
#   • Para f0 muy alta (ej. 4000 Hz), N=int(round(11.025-0.5))=10 ya tiene errores de cuantización
#     visibles. La solución real es interpolación fraccional del delay (Jaffe-Smith 1983), pero NO la
#     enseñamos hoy — solo mencionarla. La Fun Task la pide opcional.
#   • Cambiar 'round' por 'int' (truncamiento) da bugs aún peores para algunas frecuencias.
#   • Compensar -0.5 sobre fs/f0 es equivalente a tomar fs/(f0) y luego restar 0.5 antes de round.
#     Asegurar que ambas formas son numéricamente lo mismo.


### 3.1 — Verifica con beating

La prueba auditiva infalible: superpón tu `ks_correct` con la sinusoide de referencia. Si está bien
afinado, **no debe haber beating** — el oído percibe un solo tono estable. Si todavía oyes pulsación,
revisa tu $N$.

In [ ]:
mix_correct = ks_correct + ref_440
save_wav('desafio10_beating_correct_vs_ref.wav', mix_correct)
save_wav('desafio10_referencia_440Hz.wav', ref_440)

plot_envelope(mix_correct, title='Mezcla KS correcto + seno 440 Hz — la envolvente debe ser plana')
play(mix_correct)


**Lo que tiene que oírse:**
- La envolvente RMS de `mix_correct` debe ser **prácticamente plana** (oscilaciones < 5%).
- Si todavía hay beating residual, casi siempre es un alumno que escribió `N = int(fs/f0 - 0.5)` (sin
  `round`) — el truncamiento introduce un sesgo que para algunas frecuencias suma medio sample en
  la otra dirección.

> [NOTA PROFESOR] Si un alumno entrega `karplus_strong_correct` con `N = int(round(fs/f0))` y *aún
> así* le suena afinado, probablemente está oyendo el seno de referencia y la cuerda KS por separado
> (los oídos las separan en streams distintos). La envolvente RMS no miente — pídeles ver la gráfica.


## 4 · El parámetro $\rho$ — qué tan larga es la cuerda 🎚️

El factor $\rho \in [0, 1]$ multiplica la salida del filtro antes de re-inyectarla al buffer. Es la
fracción de energía que sobrevive cada vuelta:

- $\rho = 0.9$: cada vuelta pierdes 10% → decae rápido (pizzicato, banjo).
- $\rho = 0.996$: pierdes 0.4% por vuelta → cuerda nylon de guitarra clásica.
- $\rho = 1.0$: **cuerda infinita** (sin pérdida). Físicamente imposible, computacionalmente trivial.

Tarea: genera tres versiones con $\rho \in \{0.9, 0.996, 1.0\}$ y compáralas auditivamente.

In [ ]:
for rho_val, label in [(0.9, 'pizzicato'), (0.996, 'nylon'), (1.0, 'infinita')]:
    audio = karplus_strong_correct(220.0, dur=3.0, rho=rho_val)
    print(f'ρ = {rho_val:.3f} ({label}):')
    plot_envelope(audio, title=f'ρ={rho_val} — {label}')
    display(play(audio))
    print()


**Discusión esperada de $\rho$:**
- $\rho=0.9$: decay de ~50 ms (pizzicato muy corto). Aún así, KS la afinación es correcta.
- $\rho=0.996$: decay de ~2 s (guitarra clásica). El "punto óptimo" del algoritmo.
- $\rho=1.0$: la cuerda no decae *por pérdida en cada vuelta*, pero el filtro promedio sigue
  removiendo agudos en cada vuelta. Resultado: la **energía total se conserva, pero migra hacia las
  bajas frecuencias**. Al final lo que queda es esencialmente un seno a $f_0$ — interesante para
  drones.

> [NOTA PROFESOR] Acá hay una conexión importante con la sesión 03 (filtros): el filtro promedio
> $H(z) = \frac{1+z^{-1}}{2}$ tiene un cero en $z=-1$ (Nyquist), o sea atenúa máximo en altas frecuencias
> y no atenúa nada la DC. Esa asimetría es lo que produce el "envejecimiento" natural del timbre.
> Si algún alumno pregunta por la respuesta en frecuencia, dibujarla en pizarra.


### 4.1 — Bonus: la escala de Do mayor (verificación de afinación masiva)

Si tu KS está bien afinado en 440, también lo está en cualquier otra frecuencia. Genera la escala mayor
de Do (8 notas) y escúchala. Si suena afinada, ganaste. Si una nota suena "raspada", revisa tu fórmula.

In [ ]:
# Frecuencias de Do mayor (C4 → C5)
notas = [261.63, 293.66, 329.63, 349.23, 392.00, 440.00, 493.88, 523.25]
nombres = ['C4', 'D4', 'E4', 'F4', 'G4', 'A4', 'B4', 'C5']

# Cada nota dura 0.6 s, con 50 ms de silencio entre notas
escala = []
for f0, nombre in zip(notas, nombres):
    note = karplus_strong_correct(f0, dur=0.7, rho=0.996)[:int(0.6 * FS)]
    silence = np.zeros(int(0.05 * FS))
    escala.append(np.concatenate([note, silence]))
escala_audio = np.concatenate(escala)

save_wav('desafio10_escala_mayor.wav', escala_audio)
play(escala_audio)


## 5 · Laboratorio abierto 🧪

KS funciona con cualquier excitación inicial — no tiene por qué ser ruido blanco. Explora **al menos
una** de estas opciones y comparte el WAV con el grupo:

| Buffer inicial | Qué timbre esperarías |
|---|---|
| Ruido blanco $\mathcal{U}(-1,1)$ | guitarra clásica (lo normal) |
| Impulso (1 muestra a 1, resto a 0) | ¿qué pasa? |
| Senoide pura de frecuencia $f_0$ | ¿cuerda "demasiado limpia"? |
| Onda cuadrada / triangular / sierra | piano? guitarra eléctrica? |
| Los primeros 5 ms de un pluck real (descarga uno) | ¿lo mejor de los dos mundos? |

La regla del juego: predice el carácter antes de oír, oye, y compáralo con tu predicción.

In [ ]:
def ks_con_buffer_inicial(buf_init, dur, fs=FS, rho=0.996):
    """KS con buffer inicial dado (no aleatorio). N viene del largo del buffer."""
    buf = np.array(buf_init, dtype=np.float64)
    N = len(buf)
    n_out = int(dur * fs)
    out = np.zeros(n_out)
    ptr = 0
    for i in range(n_out):
        out[i] = buf[ptr]
        next_sample = rho * 0.5 * (buf[ptr] + buf[(ptr - 1) % N])
        buf[ptr] = next_sample
        ptr = (ptr + 1) % N
    return out

# Comparación rápida de excitaciones distintas a la misma f0=220 Hz
N = int(round(FS / 220.0 - 0.5))

excitaciones = {
    'ruido_blanco':  np.random.uniform(-1, 1, N),
    'impulso':       np.concatenate([[1.0], np.zeros(N - 1)]),
    'cuadrada':      np.sign(np.sin(2 * np.pi * np.arange(N) / N)),
    'senoide':       np.sin(2 * np.pi * np.arange(N) / N),
}

for nombre, buf_init in excitaciones.items():
    print(f'\n=== Buffer inicial: {nombre} ===')
    audio = ks_con_buffer_inicial(buf_init, dur=2.0)
    save_wav(f'desafio10_explorar_{nombre}.wav', audio)
    plot_spec(audio, title=f'Espectro inicial — {nombre}', fmax=3000)
    display(play(audio))

# [NOTA PROFESOR] Lo que deben descubrir:
#   • Ruido blanco: timbre clásico de guitarra (todos los modos excitados desde el inicio).
#   • Impulso: igual que ruido blanco, pero más "limpio" — un impulso tiene espectro plano
#     igual que ruido blanco (de hecho, es la respuesta al impulso del lazo de KS).
#   • Senoide: el algoritmo no puede meter armónicos que no estaban — sale un seno que decae lento.
#     "El filtro solo puede quitar, no inventar." Conexión directa con S08 substractiva.
#   • Cuadrada: timbre intermedio — más brillante al inicio, decae a senoide.
#
#   La frase para sembrar: "el algoritmo KS no es un oscilador, es un sistema que sostiene
#   armónicos que la excitación ya tenía. Por eso le importa con qué lo iniciaste."


## 6 · Cierre — guía de cierre para el profesor 🎁

**Última intervención (2 min):**
- Recalcar el bug del +½ como caso particular de un principio general: **todo filtro recursivo agrega
  delay de grupo**. KS lo hace visible porque la frecuencia se calcula directamente del delay total.
- Conectar con sesión 03 (filtros): el filtro promedio es el LP IIR más simple posible. Aquí el filtro
  no es decorativo — es lo que hace que el sonido decaiga "como una cuerda" en vez de "como un loop".
- Anticipar S11 (MIR): la pregunta inversa — "dado un audio, ¿qué $f_0$ tiene?". Hoy generamos; la
  próxima semana extraemos.

**Errores conceptuales a corregir si aparecen:**

| Error que dicen | Corrección |
|---|---|
| "El bug es el `np.roll` ineficiente" | No, es correcto funcionalmente. El bug es el cálculo de $N$. |
| "Hay que sacar el filtro para que afine" | No: sin filtro, no hay decay → no es KS, es un loop de ruido. |
| "ρ=1 ⇒ cuerda eterna ⇒ peligro de overflow" | No: el filtro promedio garantiza estabilidad (cero en Nyquist, ganancia DC = 1). El nivel global se conserva. |
| "El buffer inicial debe ser aleatorio sí o sí" | No: cualquier excitación con energía sobre los modos del lazo sirve. Ruido blanco solo es la elección que excita *todos* los modos por igual. |

**Pregunta provocadora para terminar la sesión:**
> "Si KS se descubrió en 1983 y es trivial de implementar, ¿por qué Yamaha esperó hasta el VL1 (1994)
> para incluir modelado físico en sintetizadores comerciales? ¿Qué tenía de difícil escalar la idea?"

Respuesta corta para la mesa: las cuerdas tienen un modelo lineal simple. Vientos y voz son no-lineales
(retroalimentación entre flujo y presión) y requieren ecuaciones diferenciales acopladas. La síntesis
de "soplado" tomó otros 10 años en madurar.
